# Ticket Analysis Walkthrough

This notebook walks step by step through the three analytical stages of this
project, calling the `itsm` library directly — the same code the CLI
(`itsm analyze`, `itsm train`, `itsm eval`) and the FastAPI service call:

1. **Cluster analysis** — discover the ticket *families* that actually exist
   in the designations, independent of the declared taxonomy
   (`itsm.analysis.clustering`).
2. **Classification** — predict the operational `category` used for routing,
   with a calibrated confidence score and an abstention rule
   (`itsm.models.classifier`).
3. **Weak-signal analysis** — detect an abnormal rise in ticket volume for a
   family *before* it becomes an obvious incident (`itsm.models.weak_signal`).

Each stage ends with the same three example tickets run through it, and the
final section chains all three into one `analyze_ticket` function.

Background and design rationale: `docs/02-analysis.md` and `docs/03-modeling.md`.

In [1]:
# Run with the project root as the working directory (so `data/`, `artifacts/`
# and `reports/` resolve to the paths the CLI uses), regardless of whether this
# notebook was launched from `notebooks/` or from the repo root.
import os
from pathlib import Path

directory = Path.cwd()
while not (directory / "pyproject.toml").exists():
    directory = directory.parent
os.chdir(directory)
print(f"working directory: {directory}")

working directory: /mnt/c/Users/AmineDIAB/projects/Thales/ITSM


In [2]:
from itsm.config import get_settings
from itsm.data.loading import load_anomaly_manifest, load_tickets

settings = get_settings()
tickets = load_tickets(settings)

print(f"dataset={settings.dataset!r}  n_tickets={len(tickets)}  categories={tickets['category'].nunique()}")
tickets[["ticket_id", "opened_at", "designation", "category", "priority"]].head()

dataset='synthetic'  n_tickets=7955  categories=12


,ticket_id,opened_at,designation,category,priority
0,INC0004154,2025-09-13 09:33:50,Purchase order stuck in the approval workflow,Business Applications,P1 - Critical
1,INC0004982,2025-09-13 12:50:34,Impossible d'ouvrir le fichier,Software & Licensing,P4 - Low
2,INC0000001,2025-09-13 17:10:51,Account locked after too many attempts on Offi...,Access & Accounts,P3 - Moderate
3,INC0000003,2025-09-14 12:12:23,Password reset required for VPN,Access & Accounts,P3 - Moderate
4,INC0006517,2025-09-14 12:18:57,POSSIBLE CAMPAGNE DE PHISHING VISANT LE SITE D...,Security,P3 - Moderate


In [3]:
# The three example tickets used throughout the notebook, run through every stage.
new_tickets = [
    "VPN disconnects every few minutes since this morning",
    "Impossible de se connecter au VPN depuis ce matin",
    "Password reset link expired, cannot log in",
]

## 1. Cluster analysis — discover ticket families

`DesignationClusterer` runs the pipeline `normalise -> vectorise (TF-IDF) ->
reduce (Truncated SVD) -> cluster (KMeans) -> label (c-TF-IDF)`. With
`n_clusters=None` it also picks *k* by silhouette score over a candidate range
instead of assuming the declared taxonomy is the right granularity — that is
the point of the exercise: to see whether it is.

In [4]:
from itsm.analysis.clustering import DesignationClusterer

designations = tickets["designation"].astype(str).tolist()

clusterer = DesignationClusterer(n_clusters=None, backend="tfidf")  # k picked by silhouette
result = clusterer.fit(designations, tickets["category"])

print(f"discovered {result.n_clusters} families (declared taxonomy: {tickets['category'].nunique()} categories)")
result.metrics

discovered 36 families (declared taxonomy: 12 categories)


{'n_clusters': 36.0,
 'noise_ratio': 0.0,
 'silhouette': 0.4315,
 'davies_bouldin': 1.3908,
 'calinski_harabasz': 307.94,
 'adjusted_rand': 0.1556,
 'normalized_mutual_info': 0.4271,
 'homogeneity': 0.5265,
 'completeness': 0.3592,
 'v_measure': 0.4271}

Read `homogeneity` / `completeness` together, not `adjusted_rand` alone:
homogeneity high and completeness lower than that means each discovered
cluster maps cleanly onto *part of* one declared category, but one category
often splits into several clusters (different symptom templates, two
languages). A low `adjusted_rand` here is a description of the taxonomy being
coarser than what users actually type, not a failure of the clustering.

In [5]:
clusters = result.to_frame()
clusters[["cluster_id", "size", "share", "name", "dominant_category", "category_purity"]].head(15)

,cluster_id,size,share,name,dominant_category,category_purity
18,18,455,0.0572,num / clavier / touches,Hardware & Workstation,0.6154
8,8,436,0.0548,jour / mise / mise jour,Hardware & Workstation,0.3486
11,11,398,0.0500,impossible / connexion impossible / connecter,Access & Accounts,0.4899
3,3,366,0.0460,mobile / mail / mail suspect,Mobile Devices,0.3224
9,9,351,0.0441,partage / acces partage / acces,Access & Accounts,0.5641
5,5,325,0.0409,application / toutes / deconnecte,Network & VPN,0.4215
16,16,324,0.0407,log / cannot log / cannot,Access & Accounts,0.4228
4,4,318,0.0400,poste / workstation / alerte,Software & Licensing,0.4245
23,23,314,0.0395,printer / imprimante / reach,Printing,0.7006
2,2,313,0.0393,station / longer / accueil,Hardware & Workstation,0.4345


### Family sizes and how they map onto the declared taxonomy

In [6]:
import pandas as pd

from itsm.app import charts

size_frame = (
    clusters[["cluster_id", "size"]]
    .rename(columns={"cluster_id": "cluster", "size": "tickets"})
    .assign(cluster=lambda d: d["cluster"].astype(str))
)
charts.category_bars(size_frame, value="tickets", label="cluster")

alt.Chart(...)

In [7]:
contingency = (
    pd.DataFrame({"cluster": result.labels, "category": tickets["category"].to_numpy()})
    .groupby(["cluster", "category"])
    .size()
    .reset_index(name="tickets")
)
contingency["share"] = contingency["tickets"] / contingency.groupby("cluster")["tickets"].transform("sum")
contingency["cluster"] = contingency["cluster"].astype(str)
charts.purity_heatmap(contingency)

alt.Chart(...)

### Assigning a new ticket to a family

`predict` reuses the fitted vectoriser/SVD/KMeans without refitting — this is
what the API calls for an incoming ticket.

In [8]:
new_cluster_labels = clusterer.predict(new_tickets)
for text, label in zip(new_tickets, new_cluster_labels, strict=True):
    summary = next(s for s in result.summaries if s.cluster_id == label)
    print(f"[cluster {label:>2}] {summary.name:35s} <- {text}")

[cluster  2] station / longer / accueil          <- VPN disconnects every few minutes since this morning
[cluster 11] impossible / connexion impossible / connecter <- Impossible de se connecter au VPN depuis ce matin
[cluster 27] reset / password / password reset   <- Password reset link expired, cannot log in


## 2. Classification — predict the routing category

`TicketClassifier` wraps a calibrated linear model over TF-IDF. The split is
**temporal** by default (train on the past, test on the most recent slice) —
harsher than a random split, and the honest estimate of how the model behaves
once deployed on tomorrow's tickets. The headline metric is **macro-F1**: the
taxonomy is heavily imbalanced, so plain accuracy would just track the largest
category.

In [9]:
from itsm.models.classifier import TicketClassifier
from itsm.pipelines.eval_classifier import evaluate, split_dataset

x_train, x_test, y_train, y_test = split_dataset(tickets, strategy="temporal")
classifier = TicketClassifier(model="linear_svc", backend="tfidf").fit(x_train, y_train)
metrics = evaluate(classifier, x_test, y_test)

{k: metrics[k] for k in ("n_test", "accuracy", "balanced_accuracy", "macro_f1", "weighted_f1", "top3_accuracy")}

{'n_test': 1591,
 'accuracy': 0.8636,
 'balanced_accuracy': 0.8502,
 'macro_f1': 0.8504,
 'weighted_f1': 0.8641,
 'top3_accuracy': 0.9265}

### Is the model actually better than guessing the majority class?

In [10]:
from itsm.pipelines.eval_classifier import compare_models

compare_models(x_train, x_test, y_train, y_test)

,model,accuracy,macro_f1,balanced_accuracy,top3_accuracy
0,complement_nb,0.8699,0.8579,0.8513,0.9271
1,linear_svc,0.8636,0.8504,0.8502,0.9265
2,logreg,0.8598,0.8476,0.8558,0.9296
3,majority,0.1810,0.0255,0.0833,0.3074


### Which categories are hardest to separate

In [11]:
pd.DataFrame([{"category": label, **scores} for label, scores in metrics["worst_classes"]])

,category,f1,precision,recall,support
0,Database & Middleware,0.7402,0.6438,0.8704,54
1,Telephony & Meeting Rooms,0.7874,0.8772,0.7143,70
2,Mobile Devices,0.7967,0.7656,0.8305,59


### Explaining a prediction

`top_features_for` is the cheapest form of model explanation available for a
linear model over TF-IDF, and the one support teams actually read: the
n-grams whose weight pushes a ticket towards a given category.

In [12]:
top_category = tickets["category"].value_counts().index[0]
pd.DataFrame(classifier.top_features_for(top_category, n=12), columns=["term", "weight"])

,term,weight
0,word__compte,0.5119
1,word__impossible vpn,0.4978
2,word__vpn erreur,0.4978
3,word__reinitialisation,0.4964
4,word__asset,0.4583
5,word__compte bloque,0.4372
6,word__bloque tentatives,0.4355
7,word__passe asset,0.4337
8,word__canno,0.4309
9,word__reset,0.4277


### Predicting with confidence, and abstaining

Routing should only be automated when the model is confident; below
`min_confidence` the ticket is flagged for human triage instead of being
routed on a guess.

In [13]:
predictions = classifier.predict_with_confidence(new_tickets, min_confidence=0.55)
for text, pred in zip(new_tickets, predictions, strict=True):
    flag = "ABSTAIN -> human triage" if pred.abstained else "auto-routed            "
    print(f"{flag}  {pred.label:22s} confidence={pred.confidence:.2f}  <- {text}")

auto-routed              Network & VPN          confidence=0.86  <- VPN disconnects every few minutes since this morning
ABSTAIN -> human triage  Network & VPN          confidence=0.38  <- Impossible de se connecter au VPN depuis ce matin
auto-routed              Access & Accounts      confidence=0.89  <- Password reset link expired, cannot log in


In [14]:
example_probabilities = pd.DataFrame(predictions[0].alternatives, columns=["label", "score"])
charts.probability_bars(example_probabilities)

alt.Chart(...)

## 3. Weak-signal analysis — catch a rising family early

`WeakSignalDetector` scores each daily ticket count against a **seasonal,
robust** baseline (same-weekday median / trimmed mean, MAD-based scale) so
that Monday is not mistaken for an anomaly and an incident already under way
does not drag its own baseline up. Two kinds of alert are raised:

* `spike` — an abrupt, already-significant departure on one day;
* `emerging` — an EWMA of positive residuals crossing a threshold: several
  mild deviations, none individually alarming, which is the actual *weak*
  signal and fires days before the spike does.

In [15]:
from itsm.models.weak_signal import WeakSignalDetector, build_count_series

counts = build_count_series(tickets, by="category", freq="D")
detector = WeakSignalDetector()
detection = detector.detect(counts)
alerts = detection.to_frame()

print(f"{len(alerts)} alerts over {counts.shape[1]} series x {counts.shape[0]} days")
alerts.head(15)

84 alerts over 12 series x 365 days


,series,period,kind,observed,expected,robust_z,poisson_p,ewma,severity
59,Network & VPN,2026-02-25,spike,23,3.17,11.145,6.125686e-13,5.734,17.202
60,Network & VPN,2026-02-26,emerging,9,3.00,3.464,3.802992e-03,5.053,15.159
58,Network & VPN,2026-02-24,spike,22,3.50,9.889,2.950038e-11,3.415,10.245
27,Business Applications,2026-09-08,spike,14,3.17,6.088,6.221956e-06,3.073,9.219
44,Email & Collaboration,2026-05-21,spike,8,1.67,4.906,3.406654e-04,2.876,8.628
28,Business Applications,2026-09-09,emerging,7,3.00,1.799,3.350854e-02,2.691,8.073
40,Email & Collaboration,2026-05-15,emerging,8,3.17,2.716,1.592174e-02,2.405,7.215
29,Business Applications,2026-09-10,emerging,5,2.00,1.349,5.265302e-02,2.288,6.864
39,Email & Collaboration,2026-05-14,emerging,7,2.00,3.372,4.533806e-03,2.272,6.816
30,Business Applications,2026-09-11,emerging,6,3.00,1.732,8.391794e-02,2.121,6.363


### Scoring the detector against the injected ground truth

The synthetic dataset ships a manifest of injected anomalies, so detection can
be scored like a classifier: recall, false-alarm rate, and — the metric that
actually separates a weak-signal detector from a post-mortem tool —
**detection delay**.

In [16]:
from itsm.models.weak_signal import evaluate_detections

truth = load_anomaly_manifest(settings)
n_series_days = int(counts.shape[0] * counts.shape[1])
evaluation = evaluate_detections(detection, truth, n_series_days=n_series_days)
{k: v for k, v in evaluation.items() if k != "per_anomaly"}

{'n_anomalies': 4,
 'n_alerts': 84,
 'detected': 3,
 'recall': 0.75,
 'precision': 0.2857,
 'f1': 0.4138,
 'false_positive_alerts': 60,
 'mean_detection_delay_days': 13.0,
 'max_detection_delay_days': 25,
 'false_alarms_per_100_series_days': 1.37}

### Looking at the most severe series

In [17]:
top_series = alerts.iloc[0]["series"]
series_diagnostics = detection.diagnostics[detection.diagnostics["series"] == top_series]
series_alerts = alerts[alerts["series"] == top_series].assign(
    period=lambda d: pd.to_datetime(d["period"])
)

print(f"series: {top_series}")
charts.anomaly_chart(series_diagnostics, series_alerts)

series: Network & VPN


alt.LayerChart(...)

In [18]:
charts.residual_chart(series_diagnostics, detector.config.ewma_threshold)

alt.LayerChart(...)

## Putting it together

A brand-new ticket goes through all three stages: which *family* does it
resemble, what *category* should it be routed to (and how confident is that),
and is that category *currently* showing a weak signal worth knowing about
before the ticket is triaged?

In [19]:
def analyze_ticket(designation: str) -> dict:
    cluster_label = int(clusterer.predict([designation])[0])
    cluster_summary = next(s for s in result.summaries if s.cluster_id == cluster_label)
    prediction = classifier.predict_with_confidence([designation])[0]
    emerging_now = not alerts[
        (alerts["series"] == prediction.label) & (alerts["kind"] == "emerging")
    ].empty
    return {
        "designation": designation,
        "family": cluster_summary.name,
        "predicted_category": prediction.label,
        "confidence": prediction.confidence,
        "abstained": prediction.abstained,
        "weak_signal_on_category": emerging_now,
    }


pd.DataFrame(analyze_ticket(text) for text in new_tickets)

,designation,family,predicted_category,confidence,abstained,weak_signal_on_category
0,VPN disconnects every few minutes since this m...,station / longer / accueil,Network & VPN,0.8608,False,True
1,Impossible de se connecter au VPN depuis ce matin,impossible / connexion impossible / connecter,Network & VPN,0.3810,True,True
2,"Password reset link expired, cannot log in",reset / password / password reset,Access & Accounts,0.8890,False,True


## Going further

* `Makefile` / CLI equivalents: `itsm analyze designations`, `itsm train
  classifier`, `itsm eval signals` — each writes its report under `reports/`
  and, for the clusterer and classifier, persists the fitted model under
  `artifacts/` via `itsm.models.artifacts.ArtifactStore`.
* `itsm app` serves an interactive Streamlit exploration of the same three
  stages (`src/itsm/app/views`); `itsm serve` exposes them over FastAPI
  (`docs/04-api.md`).
* `docs/06-results.md` summarises the headline numbers referenced above on the
  full reference run.